# Sprint 11 · Webinar 36 · BI (Power BI + Tableau) · Student Version  
## Proyecto Inmobiliario · Construcción guiada del dashboard

**Modalidad:** Práctica guiada  
**Duración estimada:** 100 minutos

En este cuaderno vas a construir evidencia de trabajo mientras desarrollas tu proyecto en **Power BI** o **Tableau**.  
La idea no es solo ejecutar pasos, sino **documentar decisiones**, **registrar hallazgos** y **dejar trazabilidad** del proceso.

> Usa este notebook como bitácora de trabajo de la sesión.


## Fecha

Completa la información de la sesión:

- **Fecha:** `____ / ____ / 2026`
- **Instructor/a:** `________________________________`
- **Herramienta elegida:** `Power BI / Tableau`
- **Nombre del estudiante:** `________________________________`


## Objetivos de la sesión

Al finalizar esta práctica, deberías poder:

1. Generar y validar los **4 datasets CSV** del proyecto inmobiliario.
2. Construir un **modelo de datos** correcto para análisis en BI.
3. Crear KPIs y visualizaciones para responder preguntas de negocio.
4. Documentar tus decisiones de modelado, medidas y diseño del dashboard.
5. Redactar un **resumen ejecutivo** con hallazgos accionables.

### Meta de hoy
Construir una primera versión funcional del dashboard y dejar evidencia clara del proceso.


## Agenda sugerida (100 minutos)

| Tiempo | Bloque | Herramienta | Qué harás | Evidencia en este notebook |
|---:|---|:---:|---|---|
| 0–10 | Setup y datasets | Python | Generar CSV y validar estructura | Checklist + notas |
| 10–25 | Modelo de datos | Power BI / Tableau | Cargar tablas y relacionarlas | Respuestas + checkpoint |
| 25–45 | Fechas y KPIs | Power BI / Tableau | Crear medidas o campos calculados | Tabla de seguimiento |
| 45–70 | Dashboard ejecutivo y comercial | Power BI / Tableau | Construir visuales clave | Registro de visuales |
| 70–85 | Cohortes / recurrencia | Power BI / Tableau | Analizar clientes y retención | Observaciones |
| 85–100 | Cierre y entrega | Ambas | Resumir hallazgos y pegar enlace | Resumen ejecutivo |


---

# Setup · Generación de los datasets del proyecto (10 min) 🐍

**Meta:** producir los archivos CSV que usarás durante la sesión en tu herramienta BI.

Antes de ejecutar, responde:

- ¿Qué tablas crees que funcionarán como **dimensiones**?  
  `__________________________________________________________`

- ¿Qué tabla crees que funcionará como **hecho**?  
  `__________________________________________________________`

- ¿Qué campos esperas usar para relacionar las tablas?  
  `__________________________________________________________`


### Imports y configuración

Ejecuta la siguiente celda primero.

### Checklist
- [ ] La celda corrió sin errores.
- [ ] El entorno quedó listo.
- [ ] Entiendo para qué sirve esta parte del notebook.

### Si aparece un error
Describe aquí el problema y cómo lo resolviste:

`__________________________________________________________________`


In [1]:
# ============================================================
# Imports y configuración — ejecuta esta celda primero
# ============================================================

import numpy as np
import pandas as pd
import os

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

OUTPUT_DIR = 'dataset_andes_capital'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Librerías cargadas.')
print('📁 Carpeta de salida:', OUTPUT_DIR)

✅ Librerías cargadas.
📁 Carpeta de salida: dataset_andes_capital


### Generador de datasets — Proyecto Inmobiliario Andes Capital

Ejecuta la siguiente celda para crear los **4 archivos CSV** del proyecto.

### Antes de ejecutar
Completa esta predicción:

| Archivo | ¿Qué información esperas encontrar? |
|---|---|
| `hecho_ventas_propiedades.csv` | `______________________________` |
| `dim_clientes.csv` | `______________________________` |
| `dim_propiedades.csv` | `______________________________` |
| `dim_fecha.csv` | `______________________________` |


In [2]:
# ============================================================
# GENERADOR DE DATASETS — Proyecto Inmobiliario Andes Capital
# Ejecuta este bloque completo para obtener los 4 CSV
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

# ── CIUDADES Y BARRIOS ──────────────────────────────────────────────────────
ciudades_barrios = {
    'Bogotá':      ['Chapinero', 'Usaquén', 'Suba', 'Kennedy', 'Teusaquillo'],
    'Medellín':    ['El Poblado', 'Laureles', 'Belén', 'Envigado', 'Sabaneta'],
    'Cali':        ['Granada', 'El Peñón', 'Ciudad Jardín', 'Chipichape', 'Centenario'],
    'Barranquilla':['El Prado', 'Manga', 'Bello Horizonte', 'Riomar', 'Alto Prado'],
    'Cartagena':   ['Bocagrande', 'Manga', 'El Cabrero', 'Crespo', 'Castillogrande'],
    'Bucaramanga': ['Cabecera', 'La Ciudadela', 'El Recreo', 'Provenza', 'Lagos'],
}
ciudades = list(ciudades_barrios.keys())

# ── 1. DIM_CLIENTES ─────────────────────────────────────────────────────────
N_CLIENTES = 500
segmentos   = ['First-time', 'Repeat', 'Investor', 'Corporate']
paises      = ['Colombia', 'México', 'Venezuela', 'Ecuador', 'Perú', 'EEUU']

ids_cliente = [f'CUST{str(i).zfill(5)}' for i in range(1, N_CLIENTES + 1)]
ciudad_cli  = rng.choice(ciudades, size=N_CLIENTES)

dim_clientes = pd.DataFrame({
    'id_cliente':         ids_cliente,
    'segmento_comprador': rng.choice(segmentos, size=N_CLIENTES,
                                     p=[0.40, 0.30, 0.20, 0.10]),
    'pais':               rng.choice(paises, size=N_CLIENTES,
                                     p=[0.70, 0.08, 0.07, 0.06, 0.05, 0.04]),
    'ciudad':             ciudad_cli,
})
dim_clientes.to_csv(f'{OUTPUT_DIR}/dim_clientes.csv', index=False)
print(f'✅ dim_clientes.csv — {len(dim_clientes)} filas')

# ── 2. DIM_PROPIEDADES ──────────────────────────────────────────────────────
N_PROPS     = 800
tipos       = ['Apartment', 'House', 'Office', 'Commercial', 'Land']
categorias  = ['Residential', 'Commercial', 'Industrial']

ids_prop  = [f'PROP{str(i).zfill(5)}' for i in range(1, N_PROPS + 1)]
ciudad_p  = rng.choice(ciudades, size=N_PROPS)
tipo_p    = rng.choice(tipos, size=N_PROPS, p=[0.40, 0.30, 0.15, 0.10, 0.05])

precios_base = {
    'Apartment': (150_000, 400_000),
    'House':     (250_000, 800_000),
    'Office':    (200_000, 600_000),
    'Commercial':(300_000, 900_000),
    'Land':      ( 80_000, 350_000),
}
precio_pub = np.array([
    rng.integers(*precios_base[t]) for t in tipo_p
], dtype=float)

barrios = [rng.choice(ciudades_barrios[c]) for c in ciudad_p]

dim_propiedades = pd.DataFrame({
    'id_propiedad':      ids_prop,
    'tipo_propiedad':    tipo_p,
    'ciudad':            ciudad_p,
    'barrio':            barrios,
    'habitaciones':      rng.integers(1, 6, size=N_PROPS),
    'tamano_m2':         rng.integers(40, 300, size=N_PROPS),
    'precio_publicado':  precio_pub.round(0),
    'categoria_propiedad': rng.choice(categorias, size=N_PROPS,
                                      p=[0.60, 0.30, 0.10]),
})
dim_propiedades.to_csv(f'{OUTPUT_DIR}/dim_propiedades.csv', index=False)
print(f'✅ dim_propiedades.csv — {len(dim_propiedades)} filas')

# ── 3. HECHO_VENTAS_PROPIEDADES ─────────────────────────────────────────────
N_VENTAS = 2_000
canales  = ['Broker', 'Digital', 'Referral', 'Direct']

# Fechas: 2022-01-01 → 2024-12-31  (3 años completos)
fecha_inicio = pd.Timestamp('2022-01-01')
fecha_fin    = pd.Timestamp('2024-12-31')
dias_total   = (fecha_fin - fecha_inicio).days

fechas_raw = pd.to_datetime(
    rng.integers(0, dias_total, size=N_VENTAS), unit='D', origin=fecha_inicio
).normalize()

idx_prop = rng.choice(N_PROPS, size=N_VENTAS, replace=True)
prop_ids  = [ids_prop[i] for i in idx_prop]
tipos_v   = [dim_propiedades.loc[i, 'tipo_propiedad'] for i in idx_prop]

descuento    = rng.uniform(0.80, 0.95, size=N_VENTAS)
precio_venta = (precio_pub[idx_prop] * descuento).round(0)

comision_por_canal = {'Broker': 0.05, 'Digital': 0.03, 'Referral': 0.04, 'Direct': 0.025}
canal_v = rng.choice(canales, size=N_VENTAS, p=[0.40, 0.30, 0.20, 0.10])
porc_com = np.array([comision_por_canal[c] + rng.uniform(-0.005, 0.005) for c in canal_v])
monto_com = (precio_venta * porc_com).round(0)

hecho = pd.DataFrame({
    'id_venta':            [f'SALE{str(i).zfill(6)}' for i in range(1, N_VENTAS + 1)],
    'fecha_venta':         fechas_raw.strftime('%Y-%m-%d'),
    'id_cliente':          rng.choice(ids_cliente, size=N_VENTAS),
    'id_propiedad':        prop_ids,
    'ciudad':              [dim_propiedades.loc[i, 'ciudad'] for i in idx_prop],
    'precio_venta':        precio_venta,
    'tipo_propiedad':      tipos_v,
    'canal_venta':         canal_v,
    'porcentaje_comision': porc_com.round(4),
    'monto_comision':      monto_com,
})
hecho.to_csv(f'{OUTPUT_DIR}/hecho_ventas_propiedades.csv', index=False)
print(f'✅ hecho_ventas_propiedades.csv — {len(hecho)} filas')

# ── 4. DIM_FECHA (tabla calendario) ─────────────────────────────────────────
# Se genera con pandas cubriendo exactamente el rango de fechas del hecho.
# Incluye todas las columnas necesarias para inteligencia de tiempo en Power BI y Tableau.

MESES_ES = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo',    4: 'Abril',
    5: 'Mayo',  6: 'Junio',   7: 'Julio',     8: 'Agosto',
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}
DIAS_ES = {
    0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves',
    4: 'Viernes', 5: 'Sábado', 6: 'Domingo'
}

todas_fechas = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='D')

dim_fecha = pd.DataFrame({'Date': todas_fechas})
dim_fecha['Año']            = dim_fecha['Date'].dt.year
dim_fecha['Mes']            = dim_fecha['Date'].dt.month.map(MESES_ES)
dim_fecha['Mes Numero']     = dim_fecha['Date'].dt.month
dim_fecha['Año-Mes']        = dim_fecha['Date'].dt.strftime('%Y-%m')
dim_fecha['Trimestre']      = 'Q' + dim_fecha['Date'].dt.quarter.astype(str)
dim_fecha['Año-Trimestre']  = dim_fecha['Date'].dt.year.astype(str) + '-Q' + dim_fecha['Date'].dt.quarter.astype(str)
dim_fecha['Dia Semana']     = dim_fecha['Date'].dt.dayofweek.map(DIAS_ES)
dim_fecha['Es Fin Semana']  = dim_fecha['Date'].dt.dayofweek >= 5
dim_fecha['Date']           = dim_fecha['Date'].dt.strftime('%Y-%m-%d')

dim_fecha.to_csv(f'{OUTPUT_DIR}/dim_fecha.csv', index=False)
print(f'✅ dim_fecha.csv — {len(dim_fecha)} filas  '
      f'({dim_fecha["Date"].min()} → {dim_fecha["Date"].max()})')

print('\n🏁 ¡Los 4 CSV están listos! Carga la carpeta dataset_andes_capital/ en Power BI o Tableau.')


✅ dim_clientes.csv — 500 filas
✅ dim_propiedades.csv — 800 filas
✅ hecho_ventas_propiedades.csv — 2000 filas
✅ dim_fecha.csv — 1096 filas  (2022-01-01 → 2024-12-31)

🏁 ¡Los 4 CSV están listos! Carga la carpeta dataset_andes_capital/ en Power BI o Tableau.


### Vista previa de los datasets

Ejecuta la siguiente celda para inspeccionar los archivos creados.

### Mientras revisas, completa:

1. El dataset con más columnas parece ser: `____________________`
2. La columna que probablemente será clave de fecha es: `____________________`
3. Una variable categórica que podría servir como filtro es: `____________________`
4. Una métrica numérica importante para negocio es: `____________________`


In [3]:
# Vista previa — ejecuta para verificar antes de ir a las herramientas BI
print('=== hecho_ventas_propiedades — primeras 5 filas ===')
display(hecho.head())

print('\n=== dim_clientes — primeras 5 filas ===')
display(dim_clientes.head())

print('\n=== dim_propiedades — primeras 5 filas ===')
display(dim_propiedades.head())

print('\n=== dim_fecha — primeras 5 filas ===')
display(dim_fecha.head())

print(f'\n📊 Rango de fechas: {hecho.fecha_venta.min()} → {hecho.fecha_venta.max()}')
print(f'📅 dim_fecha cubre: {dim_fecha["Date"].min()} → {dim_fecha["Date"].max()} ({len(dim_fecha)} días)')
print(f'💰 Ingreso total: ${hecho.precio_venta.sum():,.0f}')
print(f'🏠 Tipos de propiedad: {sorted(hecho.tipo_propiedad.unique())}')
print(f'📢 Canales de venta: {sorted(hecho.canal_venta.unique())}')


=== hecho_ventas_propiedades — primeras 5 filas ===


,id_venta,fecha_venta,id_cliente,id_propiedad,ciudad,precio_venta,tipo_propiedad,canal_venta,porcentaje_comision,monto_comision
0,SALE000001,2022-02-08,CUST00348,PROP00545,Medellín,368761.0,House,Broker,0.0495,18263.0
1,SALE000002,2022-08-09,CUST00356,PROP00796,Cartagena,244126.0,House,Broker,0.0492,12012.0
2,SALE000003,2023-02-10,CUST00087,PROP00678,Medellín,347815.0,Apartment,Broker,0.0511,17762.0
3,SALE000004,2024-03-11,CUST00486,PROP00691,Bogotá,146769.0,Apartment,Digital,0.0322,4733.0
4,SALE000005,2022-03-23,CUST00496,PROP00081,Cali,690200.0,Commercial,Digital,0.0340,23439.0



=== dim_clientes — primeras 5 filas ===


,id_cliente,segmento_comprador,pais,ciudad
0,CUST00001,Repeat,Colombia,Bogotá
1,CUST00002,Investor,Colombia,Cartagena
2,CUST00003,Repeat,Colombia,Barranquilla
3,CUST00004,First-time,Colombia,Cali
4,CUST00005,First-time,Colombia,Cali



=== dim_propiedades — primeras 5 filas ===


,id_propiedad,tipo_propiedad,ciudad,barrio,habitaciones,tamano_m2,precio_publicado,categoria_propiedad
0,PROP00001,Apartment,Cartagena,Crespo,1,238,236319.0,Industrial
1,PROP00002,Apartment,Bogotá,Usaquén,5,117,314333.0,Commercial
2,PROP00003,Apartment,Cartagena,Manga,3,285,305357.0,Residential
3,PROP00004,Apartment,Bogotá,Suba,3,94,330054.0,Residential
4,PROP00005,Commercial,Bogotá,Usaquén,3,276,413544.0,Commercial



=== dim_fecha — primeras 5 filas ===


,Date,Año,Mes,Mes Numero,Año-Mes,Trimestre,Año-Trimestre,Dia Semana,Es Fin Semana
0,2022-01-01,2022,Enero,1,2022-01,Q1,2022-Q1,Sábado,True
1,2022-01-02,2022,Enero,1,2022-01,Q1,2022-Q1,Domingo,True
2,2022-01-03,2022,Enero,1,2022-01,Q1,2022-Q1,Lunes,False
3,2022-01-04,2022,Enero,1,2022-01,Q1,2022-Q1,Martes,False
4,2022-01-05,2022,Enero,1,2022-01,Q1,2022-Q1,Miércoles,False



📊 Rango de fechas: 2022-01-01 → 2024-12-30
📅 dim_fecha cubre: 2022-01-01 → 2024-12-31 (1096 días)
💰 Ingreso total: $704,112,074
🏠 Tipos de propiedad: ['Apartment', 'Commercial', 'House', 'Land', 'Office']
📢 Canales de venta: ['Broker', 'Digital', 'Direct', 'Referral']


---

# Parte 1 · Modelo de datos (15 min)

En esta parte vas a construir la base analítica del proyecto.  
No se trata solo de conectar tablas: debes verificar que el modelo tenga sentido para responder preguntas de negocio.

## 1.1 · Fundamentos del esquema estrella

Con tus palabras, responde:

- Un **esquema estrella** es:  
  `______________________________________________________________`

- ¿Por qué es útil en BI?  
  `______________________________________________________________`

- En este proyecto, la tabla central debería ser:  
  `______________________________________________________________`

### Identificación del modelo
Completa la siguiente tabla:

| Tabla | ¿Hecho o dimensión? | Clave principal probable | ¿Qué describe o registra? |
|---|---|---|---|
| `hecho_ventas_propiedades` | `__________` | `__________` | `____________________` |
| `dim_clientes` | `__________` | `__________` | `____________________` |
| `dim_propiedades` | `__________` | `__________` | `____________________` |
| `dim_fecha` | `__________` | `__________` | `____________________` |


## 1.2 · Construcción del modelo en Power BI

Sigue estos pasos en **Power BI Desktop** y ve marcando lo que completes.

### Carga de archivos
- [ ] Abrí Power BI Desktop.
- [ ] Importé `hecho_ventas_propiedades.csv`.
- [ ] Importé `dim_clientes.csv`.
- [ ] Importé `dim_propiedades.csv`.
- [ ] Importé `dim_fecha.csv`.

### Validación de tipos de dato
Registra aquí los tipos que revisaste o corregiste:

| Tabla | Campo | Tipo correcto | ¿Tuve que corregirlo? |
|---|---|---|---|
| `hecho_ventas_propiedades` | `fecha_venta` | Fecha | `Sí / No` |
| `hecho_ventas_propiedades` | `precio_venta` | Decimal | `Sí / No` |
| `hecho_ventas_propiedades` | `monto_comision` | Decimal | `Sí / No` |
| `dim_fecha` | `Date` | Fecha | `Sí / No` |
| `dim_fecha` | `Año` | Entero | `Sí / No` |

### Relaciones
Anota las relaciones que creaste:

1. `______________________________________________________________`
2. `______________________________________________________________`
3. `______________________________________________________________`

### Checkpoint
Describe cómo quedó tu modelo:

`__________________________________________________________________`

> Debe verse como una estrella, con la tabla de hechos al centro y las dimensiones alrededor.


## 1.3 · Construcción del modelo en Tableau

Si trabajas en **Tableau Desktop**, completa esta parte.

### Carga de archivos
- [ ] Abrí Tableau Desktop.
- [ ] Cargué `hecho_ventas_propiedades.csv`.
- [ ] Relacioné `dim_clientes.csv`.
- [ ] Relacioné `dim_propiedades.csv`.
- [ ] Revisé los tipos de dato de las métricas y la fecha.

### Registro del modelo
Completa:

- Campo usado para relacionar clientes: `________________________`
- Campo usado para relacionar propiedades: `______________________`
- Campo de fecha principal: `_______________________________`

### Reflexión
¿Cuál es una diferencia importante entre el modelo en Power BI y en Tableau?

`__________________________________________________________________`


In [ ]:
# Espacio del estudiante
# Usa esta celda para anotar incidencias del modelo,
# capturas que debas describir, o validaciones realizadas.



---

# Parte 2 · Tabla de fechas y validación temporal (10 min)

## 2.1 · ¿Por qué es importante una tabla de fechas?

Responde:

- Una tabla de fechas permite:  
  `______________________________________________________________`

- Sin una buena dimensión de fechas, podría fallar:  
  `______________________________________________________________`

- Dos columnas temporales útiles en análisis suelen ser:  
  `______________________________________________________________`

## 2.2 · Trabajo en Power BI
Si usas Power BI:

- [ ] Seleccioné la tabla `dim_fecha`.
- [ ] La marqué como tabla de fechas.
- [ ] Elegí la columna `Date`.
- [ ] Verifiqué que la relación con `fecha_venta` esté activa.

### Checkpoint de validación
¿Cómo comprobaste que el filtro por tiempo sí funciona?

`__________________________________________________________________`

## 2.3 · Trabajo en Tableau
Si usas Tableau:

- [ ] Verifiqué que `fecha_venta` esté como **Date**.
- [ ] Probé funciones de año, mes o truncamiento de fecha.
- [ ] Confirmé que puedo construir análisis temporal sin `dim_fecha` como tabla física obligatoria.

### Nota del estudiante
¿Qué te resultó más claro o más confuso de este paso?

`__________________________________________________________________`


In [ ]:
# Espacio del estudiante
# Escribe aquí observaciones sobre filtros de fecha,
# problemas con granularidad o validaciones temporales.



---

# Parte 3 · KPIs y medidas / campos calculados (20 min)

En esta parte debes construir las métricas que alimentarán tu dashboard.

## 3.1 · KPIs base

Define con tus palabras qué significa cada KPI para este proyecto:

| KPI | ¿Qué mide? | ¿Por qué importa? |
|---|---|---|
| Ingreso Total | `__________________` | `__________________` |
| Cantidad de Ventas | `__________________` | `__________________` |
| Ticket Promedio | `__________________` | `__________________` |
| Comisión Total | `__________________` | `__________________` |


## 3.2 · Registro de medidas en Power BI

Si trabajas en Power BI, crea las medidas necesarias y completa esta tabla.

| Medida | ¿La creé? | Formato aplicado | Dificultad (1–5) | Nota |
|---|---|---|---:|---|
| `Ingreso Total` | `Sí / No` | `________` | `__` | `________` |
| `Cantidad Ventas` | `Sí / No` | `________` | `__` | `________` |
| `Ticket Promedio` | `Sí / No` | `________` | `__` | `________` |
| `Comision Total` | `Sí / No` | `________` | `__` | `________` |
| `Ingreso Apartamentos` | `Sí / No` | `________` | `__` | `________` |
| `Ingreso Broker` | `Sí / No` | `________` | `__` | `________` |
| `% del Total Ingreso` | `Sí / No` | `________` | `__` | `________` |
| `Ingreso YTD` | `Sí / No` | `________` | `__` | `________` |
| `Ingreso Año Anterior` | `Sí / No` | `________` | `__` | `________` |
| `YoY %` | `Sí / No` | `________` | `__` | `________` |
| `Ingreso Mes Anterior` | `Sí / No` | `________` | `__` | `________` |
| `MoM %` | `Sí / No` | `________` | `__` | `________` |

### Preguntas de comprensión
- ¿Qué hace `CALCULATE` en términos generales?  
  `______________________________________________________________`

- ¿Qué cambia cuando usas `ALL()`?  
  `______________________________________________________________`


## 3.3 · Registro de campos calculados en Tableau

Si trabajas en Tableau, completa esta tabla.

| Campo calculado | ¿Lo creé? | Tipo | Dificultad (1–5) | Nota |
|---|---|---|---:|---|
| `Ingreso Total` | `Sí / No` | `Medida` | `__` | `________` |
| `Cantidad de Ventas` | `Sí / No` | `Medida` | `__` | `________` |
| `Ticket Promedio` | `Sí / No` | `Medida` | `__` | `________` |
| `Comisión Total` | `Sí / No` | `Medida` | `__` | `________` |
| `Total General` | `Sí / No` | `LOD` | `__` | `________` |
| `% Participación` | `Sí / No` | `Medida` | `__` | `________` |
| `Primera Compra` | `Sí / No` | `LOD` | `__` | `________` |
| `Cohorte (Año-Mes)` | `Sí / No` | `Dimensión` | `__` | `________` |

### Preguntas de comprensión
- Una expresión **LOD** te sirve cuando:  
  `______________________________________________________________`

- La diferencia entre una medida simple y una LOD es:  
  `______________________________________________________________`


## 3.4 · Validación de resultados

Después de crear tus medidas/cálculos, responde:

1. ¿Cuál KPI te dio el valor más alto o más llamativo?  
   `__________________________________________________________`

2. ¿Qué métrica te costó más construir?  
   `__________________________________________________________`

3. ¿Qué validación hiciste para asegurarte de que los valores fueran razonables?  
   `__________________________________________________________`


In [ ]:
# Espacio del estudiante
# Puedes pegar aquí borradores de DAX, nombres de campos calculados,
# o notas sobre errores que encontraste al crear KPIs.



---

# Parte 4 · Dashboard en Power BI (25 min)

Completa esta sección si trabajas en **Power BI**.

## 4.1 · Página 1: Overview Ejecutivo

### Planeación
Antes de construir la página, define:

- Público objetivo del dashboard: `________________________________`
- Decisión que esta página debería facilitar: `______________________`

### Visuales a construir
Marca cuando termines cada uno:

- [ ] 4 tarjetas KPI
- [ ] Tendencia mensual de ingresos
- [ ] Ingresos por ciudad
- [ ] Indicador o tarjeta de crecimiento YoY
- [ ] Segmentadores/filtros

### Registro de construcción
| Visual | Campos usados | ¿Qué responde? | ¿Funcionó bien? |
|---|---|---|---|
| Tarjetas KPI | `________________` | `________________` | `Sí / No` |
| Línea mensual | `________________` | `________________` | `Sí / No` |
| Barras por ciudad | `________________` | `________________` | `Sí / No` |
| Filtro por año | `________________` | `________________` | `Sí / No` |

### Observación
¿Cuál visual fue el más útil en esta página y por qué?

`__________________________________________________________________`


## 4.2 · Página 2: Análisis Comercial

Construye una página orientada a entender qué combinaciones de producto, canal y segmento aportan más valor.

### Visuales esperados
- [ ] Barras por tipo de propiedad
- [ ] Barras por canal de venta
- [ ] Barras por segmento de comprador
- [ ] Tabla de detalle
- [ ] Algún elemento opcional adicional

### Evidencia de análisis
Completa:

- El tipo de propiedad con mayor ingreso fue: `______________________`
- El canal con mejor desempeño fue: `_____________________________`
- El segmento de comprador con más actividad fue: `________________`

### Diseño
¿Qué cambio harías para mejorar la legibilidad de esta página?

`__________________________________________________________________`


## 4.3 · Página 3: Cohortes y recurrencia

### Antes de construir
Explica con tus palabras:

- Una **cohorte** es:  
  `______________________________________________________________`

- La **recurrencia** es importante porque:  
  `______________________________________________________________`

### Desarrollo
Marca lo realizado:
- [ ] Creé la lógica para identificar la primera compra por cliente.
- [ ] Construí la dimensión o campo de cohorte.
- [ ] Creé una matriz / heatmap / visual equivalente.
- [ ] Mostré KPIs de clientes únicos y recurrentes.

### Hallazgos iniciales
- La cohorte o grupo que más te llamó la atención fue: `________________`
- Un posible patrón de recompra que observaste fue: `________________`
- Una duda que aún tienes sobre esta visual es: `___________________`


In [ ]:
# Espacio del estudiante
# Describe aquí cómo organizaste las páginas del dashboard en Power BI
# o qué ajustes de formato/interactividad aplicaste.



---

# Parte 5 · Dashboard en Tableau (20 min)

Completa esta sección si trabajas en **Tableau**.

## 5.1 · Verificación del modelo

Marca lo realizado:
- [ ] Verifiqué la relación entre tablas.
- [ ] Confirmé el tipo fecha de `fecha_venta`.
- [ ] Revisé que las dimensiones y métricas estén bien clasificadas.

### Notas
¿Qué fue lo más distinto respecto a Power BI?

`__________________________________________________________________`


## 5.2 · KPIs base en Tableau

Después de crear tus campos calculados, registra:

| KPI / campo | ¿Está listo? | ¿Dónde lo usarás? | Nota |
|---|---|---|---|
| Ingreso Total | `Sí / No` | `____________` | `________` |
| Cantidad de Ventas | `Sí / No` | `____________` | `________` |
| Ticket Promedio | `Sí / No` | `____________` | `________` |
| Comisión Total | `Sí / No` | `____________` | `________` |
| % Participación | `Sí / No` | `____________` | `________` |
| Primera Compra | `Sí / No` | `____________` | `________` |
| Cohorte (Año-Mes) | `Sí / No` | `____________` | `________` |


## 5.3 · LOD y análisis de detalle

Responde:

- ¿Para qué te sirvió el campo `Total General` o una lógica similar?  
  `______________________________________________________________`

- ¿Qué ventaja tiene calcular la primera compra por cliente con una LOD?  
  `______________________________________________________________`

- ¿Qué te resultó más retador de Tableau en esta parte?  
  `______________________________________________________________`


## 5.4 · Inteligencia de tiempo en Tableau

Marca lo que probaste:
- [ ] Variación interanual (YoY)
- [ ] Total acumulado / YTD
- [ ] Comparación mes a mes
- [ ] Filtros temporales

### Reflexión
¿Cuál fue la principal diferencia entre construir lógica temporal en Tableau vs. Power BI?

`__________________________________________________________________`


In [ ]:
# Espacio del estudiante
# Usa esta celda para anotar nombres de hojas, campos calculados,
# pruebas de filtros o decisiones de diseño en Tableau.



---

# Parte 6 · Hojas, visuales y ensamblaje final (10 min)

## 6.1 · Inventario de visuales construidos

Completa con los nombres reales de tus hojas / páginas / visuales:

| Nombre del visual / hoja | Tipo | Herramienta | ¿Qué pregunta responde? |
|---|---|---|---|
| `__________________` | `________` | `________` | `________________________` |
| `__________________` | `________` | `________` | `________________________` |
| `__________________` | `________` | `________` | `________________________` |
| `__________________` | `________` | `________` | `________________________` |
| `__________________` | `________` | `________` | `________________________` |

## 6.2 · Revisión de calidad visual

Marca si tu entrega cumple lo siguiente:

- [ ] Los títulos explican qué muestra cada visual.
- [ ] Los formatos de moneda y porcentaje son correctos.
- [ ] Los colores no dificultan la lectura.
- [ ] Los filtros funcionan correctamente.
- [ ] El dashboard responde preguntas reales de negocio.
- [ ] Hay consistencia visual entre páginas / hojas.

## 6.3 · Autoevaluación rápida del dashboard

Califica de 1 a 5:

- Claridad del modelo: `__`
- Calidad de las métricas: `__`
- Calidad visual: `__`
- Facilidad de navegación: `__`
- Confianza en mis resultados: `__`


In [ ]:
# Boceto textual del dashboard final
# Completa este espacio con una descripción breve de tu entrega.

# Herramienta:
# Nombre del dashboard:
# Páginas / hojas:
# KPIs principales:
# Filtros:
# Insight principal:
# Recomendación principal:



---

# Cierre · Resumen Ejecutivo y Entrega (10 min)

## Guía para redactar el resumen ejecutivo

Usa tu dashboard para completar este borrador:

### Hallazgos clave
- El ingreso total del período analizado fue de `________________________`.
- El tipo de propiedad con mayor peso en el negocio fue `________________`.
- La ciudad con mejor desempeño fue `_______________________________`.
- El canal más relevante según tus datos fue `________________________`.
- El comportamiento temporal más importante que observé fue `_____________`.

### Insights accionables
1. `______________________________________________________________`
2. `______________________________________________________________`
3. `______________________________________________________________`

### Recomendación estratégica principal
`__________________________________________________________________`


## Entrega

Completa la celda final con el enlace de tu proyecto.

### Formatos válidos
- **Power BI:** enlace publicado o enlace al archivo `.pbix`
- **Tableau:** enlace en Tableau Public
- **Drive / OneDrive:** enlace con permisos de visualización

### Antes de entregar
- [ ] Verifiqué que el enlace abre correctamente.
- [ ] El dashboard tiene títulos claros.
- [ ] Mis visuales y métricas están completos.
- [ ] El resumen ejecutivo coincide con mis resultados.


In [ ]:
# Enlace del proyecto (pega aquí tu link)
link_proyecto = ""

# Reflexión final breve
aprendizaje_clave = ""
siguiente_mejora = ""

print("🚀 Proyecto entregado:", link_proyecto if link_proyecto else "Aún no agregado")
print("🧠 Aprendizaje clave:", aprendizaje_clave if aprendizaje_clave else "Pendiente")
print("🔧 Siguiente mejora:", siguiente_mejora if siguiente_mejora else "Pendiente")


## Cierre personal

Completa:

- **Lo que mejor entendí hoy fue:**  
  `______________________________________________________________`

- **Lo que todavía necesito practicar es:**  
  `______________________________________________________________`

- **Mi nivel de confianza para explicar este dashboard es:**  
  `Bajo / Medio / Alto`


### ¿Necesitas ayuda?
Recuerda los canales de ayuda que tenemos disponibles:
- `DATA-CO-LEARNING`: Revisa los horarios de atencion en la guia dele studiante. Recuerda que tenemos horarios de apoyo todos los días para tus dudas puntuales!
- `DA_CONSULTA`: Uuedes publicar tus preguntas sobre el contenido de la plataforma o el proyecto 24/7. Recuerda que en tu pregunta debes de agregar el tag @Dataconsulta.
- `SPRINT FOCUS`: Para los Sprints 1 al 5 tenemos sesiones extra donde abordamos temáticas de cada sprint a rpofundidad. Revisa la guia del estudiante para conocer los horarios!
- `SESIONES 1:1`: Recuerda que puedes agendar sesiones 1:1 con un tutor. Agendalas con antelación y resuelve todas tus dudas, desde temás del proyecto, curso, carrera hasta problemas técnicos.
- `CANAL DE DISCORD DE COMMUNITY`: Recuerda que tu cohorte tiene un canal especial donde puedes compartir cualquier item interesante que quieras mostrar a tus compañeros de curso.

## Notas finales del estudiante

Usa este espacio para apuntes libres, fórmulas, ideas de mejora, dudas o recordatorios.

`____________________________________________________________________`

`____________________________________________________________________`

`____________________________________________________________________`
